# Funding Eligibility Finder

## Notebook 3 — Eligibility Scoring Engine

### Objective

This notebook takes the structured funding dataset generated by Notebook 2 and ranks each opportunity according to its suitability for the applicant profile.

The goal is to transform extracted web information into an actionable shortlist of opportunities.

---

### Input

`funding_extracted_opportunities.csv`

Generated by Notebook 2.

---

### Output

- `results/funding_ranked_opportunities.csv`
- `results/funding_ranked_opportunities.xlsx`
- `results/funding_actionable_shortlist.csv`
- `results/funding_actionable_shortlist.xlsx`

---

### Workflow

1. Upload the extracted opportunities file
2. Load applicant profile
3. Score eligibility match
4. Score field relevance
5. Score funding signal
6. Score amount
7. Score deadline
8. Score application readiness
9. Penalize restrictions and noisy pages
10. Generate final recommendation
11. Export ranked results and actionable shortlist

In [1]:
!pip install -q pandas openpyxl dateparser

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.5/300.5 kB 12.3 MB/s eta 0:00:00


In [8]:
# ============================================================
# LOAD INPUT FILE FROM NOTEBOOK 2
# ============================================================

from google.colab import files
import pandas as pd

print("Upload the file generated by Notebook 2:")
print("funding_extracted_opportunities.csv\n")

uploaded = files.upload()

if len(uploaded) == 0:
    raise FileNotFoundError("No file was uploaded.")

INPUT_FILE = next(iter(uploaded.keys()))

print(f"\nInput file detected: {INPUT_FILE}")

input_check = pd.read_csv(INPUT_FILE)

print(f"Loaded {len(input_check)} extracted opportunities.")
print(f"Dataset shape: {input_check.shape}")

display(input_check.head())

Upload the file generated by Notebook 2:
funding_extracted_opportunities.csv



Saving funding_extracted_opportunities.csv to funding_extracted_opportunities (1).csv

Input file detected: funding_extracted_opportunities (1).csv
Loaded 60 extracted opportunities.
Dataset shape: (60, 25)


,query,title,url,snippet,status,relevance_score,is_aggregator,funding_matches,profile_matches,negative_flags,...,final_url,fetch_error,extracted_text_length,possible_amounts,amount_snippets,deadline_snippets,parsed_deadlines,eligibility_snippets,application_links,page_type_signal
0,female finance students scholarship,Top 2026 Scholarships for Women: College Fundi...,https://www.fastweb.com/college-scholarships/a...,Explore the best 2026 scholarships for women a...,high potential,27,False,"scholarship, scholarships, grant, grants, fund...","women, female, stem",NaN,...,https://www.fastweb.com/college-scholarships/a...,NaN,19742,"$5,000, $10,000, $15,000, $3,000, $2,000, $25,...",Available to: High School Seniors Award Amount...,Deadline: 3/15/26 Award Amount: Varies The Kri...,NaN,"In honor of Women's History Month, we are high...",https://www.fastweb.com/student-news/articles/...,unclear
1,women in financial mathematics scholarship,Funding & Scholarships for Women in Mathematic...,https://www.mathunion.org/cwm/resources/fundin...,Explore funding opportunities and scholarships...,high potential,25,False,"scholarship, scholarships, grant, grants, fund...",women,NaN,...,https://www.mathunion.org/cwm/resources/fundin...,NaN,5796,"$60,000",For example: The L’Oréal USA For Women In Scie...,NaN,NaN,Here is some information about funding that re...,https://cws.auburn.edu/shared/files?id=217&fil...,unclear
2,funding award for Italian students in data sci...,Spärck AI Scholarship | UCL Scholarships and f...,https://www.ucl.ac.uk/scholarships/sparck-ai-s...,The Spärck AI Scholarship may not be held alon...,high potential,23,False,"scholarship, scholarships, bursary, bursaries,...","master, uk",NaN,...,https://www.ucl.ac.uk/scholarships/sparck-ai-s...,NaN,5424,"£22,780, full tuition fee, tuition fees, stipend",- The value of the scholarship is full tuition...,| Programme | Application submission email add...,NaN,Eligibility Candidates must fulfil the followi...,https://view.officeapps.live.com/op/view.aspx?...,possibly specific opportunity page
3,scholarships for female mathematicians,AMS :: Fellowships & Scholarships - American M...,https://www.ams.org/grants-awards/ams-fellowsh...,The Joan and Joseph Birman Fellowship for the ...,high potential,22,False,"scholarship, scholarships, grant, grants, awar...",women,NaN,...,https://www.ams.org/grants-awards/ams-fellowsh...,NaN,2855,NaN,NaN,NaN,NaN,- Joan and Joseph Birman Fellowship for the Ad...,https://ebus.ams.org/ebus/Register.aspx?_gl=1*...,unclear
4,funding for Italian students pursuing a master...,Scholarships for Master's Studies in the Unite...,https://www.educations.com/articles-and-advice...,The UK Research and Innovation (UKRI) offers v...,high potential,20,False,"scholarship, scholarships, grant, grants, funding","international students, postgraduate, master, ...",NaN,...,https://www.educations.com/articles-and-advice...,NaN,4888,"£18,180, £1,515, £5,000, £15,609, £10,000, £1,...",Scholarships are a great way to make your stud...,NaN,NaN,Scholarships for Master's Studies in the Unite...,https://www.educations.com/articles-and-advice...,unclear


In [9]:
# ============================================================
# FUNDING ELIGIBILITY FINDER
# Notebook 3: Eligibility Scoring Engine
# ============================================================

import re
from pathlib import Path
from datetime import date

import numpy as np
import pandas as pd
import dateparser


# ============================================================
# CONFIGURATION
# ============================================================

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

OUTPUT_CSV = RESULTS_DIR / "funding_ranked_opportunities.csv"
OUTPUT_EXCEL = RESULTS_DIR / "funding_ranked_opportunities.xlsx"

SHORTLIST_CSV = RESULTS_DIR / "funding_actionable_shortlist.csv"
SHORTLIST_EXCEL = RESULTS_DIR / "funding_actionable_shortlist.xlsx"

CURRENT_DATE = date.today()


# ============================================================
# APPLICANT PROFILE
# ============================================================

PROFILE = {
    "nationality": "Italian",
    "residence_country": "Italy",
    "gender": "female",
    "degree_level": "MSc",
    "study_level": "postgraduate",
    "destination_country": "UK",
    "start_year": 2026,
    "fields": [
        "financial mathematics",
        "quantitative finance",
        "mathematical finance",
        "data science",
        "finance",
        "STEM"
    ],
    "target_universities": [
        "UCL",
        "LSE"
    ]
}


# ============================================================
# KEYWORD DICTIONARIES
# ============================================================

ELIGIBILITY_KEYWORDS = {
    "gender_match": [
        "women", "woman", "female", "women in finance", "women in stem"
    ],
    "level_match": [
        "msc", "master", "masters", "master's", "postgraduate",
        "graduate", "taught postgraduate"
    ],
    "international_match": [
        "international students", "overseas students", "non-uk students",
        "eu students", "european students", "italian students",
        "students from italy"
    ],
    "destination_match": [
        "uk", "united kingdom", "london", "study in the uk",
        "uk university"
    ]
}

FIELD_KEYWORDS = [
    "financial mathematics",
    "quantitative finance",
    "mathematical finance",
    "computational finance",
    "finance",
    "economics",
    "data science",
    "stem",
    "mathematics",
    "statistics",
    "machine learning",
    "computer science"
]

FUNDING_KEYWORDS = [
    "scholarship", "scholarships",
    "grant", "grants",
    "bursary", "bursaries",
    "award", "awards",
    "funding",
    "financial aid",
    "sponsorship",
    "tuition fee",
    "tuition fees",
    "stipend",
    "maintenance support",
    "living expenses"
]

APPLICATION_KEYWORDS = [
    "apply",
    "application",
    "application form",
    "submit",
    "register",
    "how to apply"
]

RESTRICTION_KEYWORDS = {
    "phd_only": [
        "phd only", "doctoral only", "doctorate only",
        "phd students only", "doctoral students only"
    ],
    "undergraduate_only": [
        "undergraduate only", "undergraduate students only",
        "bachelor only", "bachelor's only"
    ],
    "uk_only": [
        "uk citizens only", "british citizens only",
        "home students only", "uk nationals only",
        "permanent residents of the uk"
    ],
    "us_only": [
        "us citizens only", "u.s. citizens only",
        "usa residents only", "united states citizens only"
    ],
    "lmics_only": [
        "low income countries only",
        "low-income countries only",
        "low and middle income countries",
        "developing countries only",
        "least developed countries",
        "lmics"
    ],
    "closed": [
        "applications closed",
        "closed for applications",
        "deadline passed",
        "no longer accepting applications"
    ]
}

NOISE_KEYWORDS = [
    "course page",
    "aggregator or list page",
    "directory",
    "list of scholarships",
    "top scholarships",
    "scholarships by country",
    "study abroad scholarships",
    "blog",
    "ranking",
    "tuition and fees",
    "tuition & fees"
]


# ============================================================
# UTILITY FUNCTIONS
# ============================================================

def safe_scalar(value, default=""):
    if isinstance(value, pd.Series):
        value = value.iloc[0]

    if pd.isna(value):
        return default

    return value


def safe_get(row, column_name, default=""):
    value = row.get(column_name, default)
    return safe_scalar(value, default)


def clean_text(text):
    text = safe_scalar(text, "")
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def find_matches(text, keywords):
    text = clean_text(text)
    return [keyword for keyword in keywords if keyword in text]


def combine_relevant_text(row):
    columns = [
        "title",
        "url",
        "final_url",
        "snippet",
        "possible_amounts",
        "amount_snippets",
        "deadline_snippets",
        "parsed_deadlines",
        "eligibility_snippets",
        "application_links",
        "page_type_signal"
    ]

    return " ".join(
        clean_text(safe_get(row, column))
        for column in columns
    )


# ============================================================
# AMOUNT PARSING
# ============================================================

def normalize_amount(amount_text):
    amount_text = str(safe_scalar(amount_text, ""))

    if amount_text.strip() == "":
        return np.nan

    values = re.findall(
        r"[\£\€\$]?\s?(\d{1,3}(?:,\d{3})*(?:\.\d+)?)",
        amount_text
    )

    numeric_values = []

    for value in values:
        try:
            number = float(value.replace(",", ""))
            numeric_values.append(number)
        except ValueError:
            pass

    if not numeric_values:
        return np.nan

    amount = max(numeric_values)

    if amount > 100000:
        return np.nan

    return amount


# ============================================================
# DEADLINE PARSING
# ============================================================

def parse_deadline_values(deadline_text):
    deadline_text = str(safe_scalar(deadline_text, ""))

    if deadline_text.strip() == "":
        return []

    raw_items = re.split(r",|\|", deadline_text)

    parsed_dates = []

    for item in raw_items:
        parsed = dateparser.parse(
            item,
            settings={
                "DATE_ORDER": "DMY",
                "PREFER_DAY_OF_MONTH": "last"
            }
        )

        if parsed:
            parsed_dates.append(parsed.date())

    return sorted(list(set(parsed_dates)))


# ============================================================
# SCORE COMPONENTS
# ============================================================

def score_eligibility(text):
    score = 0
    reasons = []

    for category, keywords in ELIGIBILITY_KEYWORDS.items():
        matches = find_matches(text, keywords)

        if matches:
            score += 10
            reasons.append(f"{category}: {', '.join(matches[:3])}")

    return min(score, 40), reasons


def score_field_relevance(text):
    matches = find_matches(text, FIELD_KEYWORDS)
    score = min(len(matches) * 4, 20)

    return score, matches


def score_funding_signal(text):
    matches = find_matches(text, FUNDING_KEYWORDS)
    score = min(len(matches) * 3, 15)

    return score, matches


def score_amount(row):
    amount_text = f"{safe_get(row, 'possible_amounts')} {safe_get(row, 'amount_snippets')}"
    amount = normalize_amount(amount_text)

    if np.isnan(amount):
        return 0, np.nan, "no reliable amount detected"

    if amount >= 20000:
        return 15, amount, "large funding amount"
    elif amount >= 10000:
        return 12, amount, "strong funding amount"
    elif amount >= 5000:
        return 9, amount, "moderate funding amount"
    elif amount >= 1000:
        return 5, amount, "small funding amount"
    else:
        return 2, amount, "very small funding amount"


def score_deadline(row):
    parsed_deadlines = safe_get(row, "parsed_deadlines")
    dates = parse_deadline_values(parsed_deadlines)

    if not dates:
        return 3, "", "no parsed deadline"

    future_dates = [d for d in dates if d >= CURRENT_DATE]

    if not future_dates:
        return 0, ", ".join(d.isoformat() for d in dates), "expired or past deadline"

    nearest = min(future_dates)
    days_left = (nearest - CURRENT_DATE).days

    if days_left <= 14:
        return 6, nearest.isoformat(), "urgent deadline"
    elif days_left <= 60:
        return 10, nearest.isoformat(), "upcoming deadline"
    elif days_left <= 180:
        return 8, nearest.isoformat(), "future deadline"
    else:
        return 5, nearest.isoformat(), "far future deadline"


def score_application(row):
    links = clean_text(safe_get(row, "application_links"))

    if links == "":
        return 0, "no application link detected"

    matches = find_matches(links, APPLICATION_KEYWORDS)

    if matches:
        return 10, "application link detected"

    return 6, "possibly useful link detected"


def score_restrictions(text):
    flags = []

    for restriction_type, keywords in RESTRICTION_KEYWORDS.items():
        if any(keyword in text for keyword in keywords):
            flags.append(restriction_type)

    penalty = -5 * len(flags)

    if "closed" in flags:
        penalty -= 10

    if "phd_only" in flags or "undergraduate_only" in flags:
        penalty -= 10

    if "uk_only" in flags or "us_only" in flags:
        penalty -= 10

    penalty = max(penalty, -35)

    return penalty, flags


def score_page_quality(row, text):
    page_type = clean_text(safe_get(row, "page_type_signal"))
    url = clean_text(safe_get(row, "url"))
    final_url = clean_text(safe_get(row, "final_url"))

    score = 0
    flags = []

    if page_type == "specific opportunity page":
        score += 10
        flags.append("specific opportunity page")

    elif page_type == "possibly specific opportunity page":
        score += 5
        flags.append("possibly specific opportunity page")

    elif page_type == "course page":
        score -= 10
        flags.append("course page")

    elif page_type == "aggregator or list page":
        score -= 12
        flags.append("aggregator or list page")

    elif page_type == "not retrieved":
        score -= 20
        flags.append("not retrieved")

    elif page_type == "low text content":
        score -= 8
        flags.append("low text content")

    if any(keyword in text for keyword in NOISE_KEYWORDS):
        score -= 5
        flags.append("noise keywords")

    official_domains = [
        ".ac.uk",
        ".edu",
        ".gov.uk",
        "ucl.ac.uk",
        "lse.ac.uk",
        "imperial.ac.uk",
        "cam.ac.uk",
        "bath.ac.uk",
        "britishcouncil",
        "gov.uk"
    ]

    if any(domain in url or domain in final_url for domain in official_domains):
        score += 5
        flags.append("official source")

    return score, flags


# ============================================================
# RECOMMENDATION ENGINE
# ============================================================

def classify_recommendation(final_score, restriction_flags, page_type_signal):
    hard_restrictions = {
        "phd_only",
        "undergraduate_only",
        "uk_only",
        "us_only",
        "lmics_only",
        "closed"
    }

    if any(flag in hard_restrictions for flag in restriction_flags):
        return "Ignore"

    if page_type_signal == "not retrieved":
        return "Review Carefully"

    if final_score >= 85:
        return "Apply Immediately"
    elif final_score >= 70:
        return "Apply"
    elif final_score >= 50:
        return "Review Carefully"
    elif final_score >= 30:
        return "Low Priority"
    else:
        return "Ignore"


def assign_next_action(row):
    recommendation = safe_get(row, "recommendation")
    nearest_deadline = safe_get(row, "nearest_deadline")
    application_links = safe_get(row, "application_links")
    restriction_flags = safe_get(row, "restriction_flags")

    if recommendation == "Apply Immediately":
        if nearest_deadline:
            return "Verify eligibility and prepare the application immediately."
        return "High-priority opportunity: verify the deadline manually and prepare materials."

    if recommendation == "Apply":
        if application_links:
            return "Review the application page and prepare a draft application."
        return "Promising opportunity: manually search for the application form."

    if recommendation == "Review Carefully":
        return "Manually verify eligibility, deadline, amount and application requirements."

    if recommendation == "Low Priority":
        return "Keep as backup opportunity; review after stronger options."

    if recommendation == "Ignore":
        if restriction_flags:
            return "Ignore due to detected restrictions or low fit."
        return "Ignore for now due to low relevance."

    return "Manual review required."


def build_reason(row):
    parts = []

    fields = {
        "Eligibility": safe_get(row, "eligibility_reasons"),
        "Field": safe_get(row, "field_matches"),
        "Funding": safe_get(row, "funding_matches"),
        "Amount": safe_get(row, "amount_reason"),
        "Deadline": safe_get(row, "deadline_reason"),
        "Application": safe_get(row, "application_reason"),
        "Restrictions": safe_get(row, "restriction_flags"),
        "Quality": safe_get(row, "page_quality_flags")
    }

    for label, value in fields.items():
        if str(value).strip():
            parts.append(f"{label}: {value}")

    return " | ".join(parts)


# ============================================================
# OPPORTUNITY SCORING
# ============================================================

def score_opportunity(row):
    text = combine_relevant_text(row)

    eligibility_score, eligibility_reasons = score_eligibility(text)
    field_score, field_matches = score_field_relevance(text)
    funding_score, funding_matches = score_funding_signal(text)
    amount_score, normalized_amount, amount_reason = score_amount(row)
    deadline_score, nearest_deadline, deadline_reason = score_deadline(row)
    application_score, application_reason = score_application(row)
    restriction_penalty, restriction_flags = score_restrictions(text)
    page_quality_score, page_quality_flags = score_page_quality(row, text)

    final_score = (
        eligibility_score
        + field_score
        + funding_score
        + amount_score
        + deadline_score
        + application_score
        + restriction_penalty
        + page_quality_score
    )

    final_score = max(min(final_score, 100), 0)

    page_type_signal = safe_get(row, "page_type_signal")

    recommendation = classify_recommendation(
        final_score,
        restriction_flags,
        page_type_signal
    )

    return pd.Series({
        "eligibility_score": eligibility_score,
        "field_score": field_score,
        "funding_signal_score": funding_score,
        "amount_score": amount_score,
        "deadline_score": deadline_score,
        "application_score": application_score,
        "restriction_penalty": restriction_penalty,
        "page_quality_score": page_quality_score,
        "final_score": final_score,
        "recommendation": recommendation,
        "normalized_amount": normalized_amount,
        "nearest_deadline": nearest_deadline,
        "eligibility_reasons": " ; ".join(eligibility_reasons),
        "field_matches": ", ".join(field_matches),
        "funding_matches": ", ".join(funding_matches),
        "restriction_flags": ", ".join(restriction_flags),
        "page_quality_flags": ", ".join(page_quality_flags),
        "amount_reason": amount_reason,
        "deadline_reason": deadline_reason,
        "application_reason": application_reason
    })


# ============================================================
# PIPELINE
# ============================================================

def run_scoring_pipeline():
    df = pd.read_csv(INPUT_FILE)

    df = df.loc[:, ~df.columns.duplicated()].copy()

    print(f"Loaded extracted opportunities: {len(df)}")

    scoring = df.apply(score_opportunity, axis=1)

    results = pd.concat([df, scoring], axis=1)

    recommendation_order = {
        "Apply Immediately": 1,
        "Apply": 2,
        "Review Carefully": 3,
        "Low Priority": 4,
        "Ignore": 5
    }

    results["recommendation_rank"] = results["recommendation"].map(
        recommendation_order
    )

    results = (
        results
        .sort_values(
            by=["recommendation_rank", "final_score"],
            ascending=[True, False]
        )
        .drop(columns=["recommendation_rank"])
        .reset_index(drop=True)
    )

    results["reason"] = results.apply(build_reason, axis=1)
    results["next_action"] = results.apply(assign_next_action, axis=1)

    results.to_csv(OUTPUT_CSV, index=False)
    results.to_excel(OUTPUT_EXCEL, index=False)

    actionable_shortlist = results[
        results["recommendation"].isin(
            ["Apply Immediately", "Apply", "Review Carefully"]
        )
    ].copy()

    actionable_shortlist.to_csv(SHORTLIST_CSV, index=False)
    actionable_shortlist.to_excel(SHORTLIST_EXCEL, index=False)

    print("\nScoring completed.")
    print(f"Saved ranked opportunities: {OUTPUT_CSV}")
    print(f"Saved actionable shortlist: {SHORTLIST_CSV}")

    return results, actionable_shortlist


# ============================================================
# RUN NOTEBOOK 3
# ============================================================

ranked_results, actionable_shortlist = run_scoring_pipeline()

display(
    ranked_results[
        [
            "recommendation",
            "final_score",
            "title",
            "normalized_amount",
            "nearest_deadline",
            "page_type_signal",
            "next_action",
            "url"
        ]
    ].head(30)
)

Loaded extracted opportunities: 60

Scoring completed.
Saved ranked opportunities: results/funding_ranked_opportunities.csv
Saved actionable shortlist: results/funding_actionable_shortlist.csv


,recommendation,final_score,title,normalized_amount,nearest_deadline,page_type_signal,next_action,url
0,Apply Immediately,98,"Taught postgraduate scholarships, bursaries, a...",20780.00,2026-12-06,possibly specific opportunity page,Verify eligibility and prepare the application...,https://www.bath.ac.uk/topics/taught-postgradu...
1,Apply Immediately,96,Artificial Intelligence and Data Science Postg...,10000.00,,possibly specific opportunity page,High-priority opportunity: verify the deadline...,https://www.gre.ac.uk/bursaries/artificial-int...
2,Apply Immediately,92,Masters scholarships for 2026/27: how to apply...,30000.00,,specific opportunity page,High-priority opportunity: verify the deadline...,https://www.prospects.ac.uk/postgraduate-study...
3,Apply Immediately,90,UCL Master’s Funding Opportunities 2026/27: Bu...,42875.00,2026-07-05,specific opportunity page,Verify eligibility and prepare the application...,https://opportunitiesforyouth.org/2026/02/17/u...
4,Apply Immediately,85,Spärck AI Scholarship | UCL Scholarships and f...,22780.00,,possibly specific opportunity page,High-priority opportunity: verify the deadline...,https://www.ucl.ac.uk/scholarships/sparck-ai-s...
5,Apply Immediately,85,Women in STEM Scholarships for Studying Overse...,NaN,,possibly specific opportunity page,High-priority opportunity: verify the deadline...,https://www.icanstudent.com/women-in-stem-scho...
6,Apply,84,Scholarships for Master's Studies in the Unite...,18744.00,,unclear,Review the application page and prepare a draf...,https://www.educations.com/articles-and-advice...
7,Apply,82,Women's Scholarship for International Students...,5000.00,,specific opportunity page,Review the application page and prepare a draf...,https://www.educations.com/scholarships/women-...
8,Apply,81,Grants Scholarships and Bursaries UK (2026),42875.00,,specific opportunity page,Review the application page and prepare a draf...,https://unisorted.co.uk/student-money/grants-s...
9,Apply,80,Scholarships for international students | Univ...,16200.00,,possibly specific opportunity page,Review the application page and prepare a draf...,https://www.port.ac.uk/study/international-stu...


In [11]:
print("Recommendation distribution:")
print(ranked_results["recommendation"].value_counts())

print("\nAverage final score:")
print(ranked_results["final_score"].mean())

print("\nActionable opportunities:")
print(len(actionable_shortlist))

print("\nTop ranked opportunities:")
display(
    ranked_results[
        [
            "recommendation",
            "final_score",
            "title",
            "normalized_amount",
            "nearest_deadline",
            "restriction_flags",
            "page_quality_flags",
            "url"
        ]
    ].head(20)
)

Recommendation distribution:
recommendation
Review Carefully     30
Apply                14
Low Priority          8
Apply Immediately     6
Ignore                2
Name: count, dtype: int64

Average final score:
56.28333333333333

Actionable opportunities:
50

Top ranked opportunities:


,recommendation,final_score,title,normalized_amount,nearest_deadline,restriction_flags,page_quality_flags,url
0,Apply Immediately,98,"Taught postgraduate scholarships, bursaries, a...",20780.00,2026-12-06,,"possibly specific opportunity page, official s...",https://www.bath.ac.uk/topics/taught-postgradu...
1,Apply Immediately,96,Artificial Intelligence and Data Science Postg...,10000.00,,,"possibly specific opportunity page, official s...",https://www.gre.ac.uk/bursaries/artificial-int...
2,Apply Immediately,92,Masters scholarships for 2026/27: how to apply...,30000.00,,,"specific opportunity page, official source",https://www.prospects.ac.uk/postgraduate-study...
3,Apply Immediately,90,UCL Master’s Funding Opportunities 2026/27: Bu...,42875.00,2026-07-05,,specific opportunity page,https://opportunitiesforyouth.org/2026/02/17/u...
4,Apply Immediately,85,Spärck AI Scholarship | UCL Scholarships and f...,22780.00,,,"possibly specific opportunity page, official s...",https://www.ucl.ac.uk/scholarships/sparck-ai-s...
5,Apply Immediately,85,Women in STEM Scholarships for Studying Overse...,NaN,,,possibly specific opportunity page,https://www.icanstudent.com/women-in-stem-scho...
6,Apply,84,Scholarships for Master's Studies in the Unite...,18744.00,,,"noise keywords, official source",https://www.educations.com/articles-and-advice...
7,Apply,82,Women's Scholarship for International Students...,5000.00,,,"specific opportunity page, official source",https://www.educations.com/scholarships/women-...
8,Apply,81,Grants Scholarships and Bursaries UK (2026),42875.00,,,specific opportunity page,https://unisorted.co.uk/student-money/grants-s...
9,Apply,80,Scholarships for international students | Univ...,16200.00,,,"possibly specific opportunity page, official s...",https://www.port.ac.uk/study/international-stu...


In [12]:
# ============================================================
# PROFESSIONAL EXCEL SUMMARY EXPORT
# ============================================================

from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter


SUMMARY_EXCEL = RESULTS_DIR / "funding_summary.xlsx"


def create_professional_summary(results):
    """
    Create a clean Excel summary designed for manual review.
    """

    summary_columns = [
        "recommendation",
        "final_score",
        "title",
        "normalized_amount",
        "nearest_deadline",
        "page_type_signal",
        "next_action",
        "url"
    ]

    summary = results[summary_columns].copy()

    summary = summary.rename(columns={
        "recommendation": "Recommendation",
        "final_score": "Score",
        "title": "Opportunity",
        "normalized_amount": "Amount",
        "nearest_deadline": "Deadline",
        "page_type_signal": "Page Type",
        "next_action": "Next Action",
        "url": "URL"
    })

    summary.to_excel(SUMMARY_EXCEL, index=False)

    return SUMMARY_EXCEL


def format_summary_excel(path):
    """
    Apply basic professional formatting to the Excel summary.
    """

    wb = load_workbook(path)
    ws = wb.active
    ws.title = "Funding Summary"

    header_fill = PatternFill(
        start_color="1F4E78",
        end_color="1F4E78",
        fill_type="solid"
    )

    header_font = Font(
        color="FFFFFF",
        bold=True
    )

    thin_border = Border(
        left=Side(style="thin", color="D9E2F3"),
        right=Side(style="thin", color="D9E2F3"),
        top=Side(style="thin", color="D9E2F3"),
        bottom=Side(style="thin", color="D9E2F3")
    )

    recommendation_fills = {
        "Apply Immediately": PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid"),
        "Apply": PatternFill(start_color="D9EAF7", end_color="D9EAF7", fill_type="solid"),
        "Review Carefully": PatternFill(start_color="FFF2CC", end_color="FFF2CC", fill_type="solid"),
        "Low Priority": PatternFill(start_color="FCE4D6", end_color="FCE4D6", fill_type="solid"),
        "Ignore": PatternFill(start_color="F4CCCC", end_color="F4CCCC", fill_type="solid")
    }

    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center")
        cell.border = thin_border

    for row in ws.iter_rows(min_row=2):
        recommendation = row[0].value

        for cell in row:
            cell.alignment = Alignment(
                vertical="top",
                wrap_text=True
            )
            cell.border = thin_border

        if recommendation in recommendation_fills:
            row[0].fill = recommendation_fills[recommendation]

    column_widths = {
        "A": 20,
        "B": 10,
        "C": 55,
        "D": 14,
        "E": 16,
        "F": 28,
        "G": 60,
        "H": 70
    }

    for column, width in column_widths.items():
        ws.column_dimensions[column].width = width

    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

    wb.save(path)


summary_path = create_professional_summary(ranked_results)
format_summary_excel(summary_path)

print(f"Professional Excel summary saved: {summary_path}")

Professional Excel summary saved: results/funding_summary.xlsx


In [13]:
from google.colab import files

files.download(str(OUTPUT_CSV))
files.download(str(SHORTLIST_CSV))
files.download(str(SUMMARY_EXCEL))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>